In [16]:
import pandas as pd

config_files = [
    "pom.xml",
    "requirements.txt",
    "setup.py",
    "pyproject.toml",
    "setup.cfg",
]
base_folder = "benchmark/Phase1"

In [15]:
def count():
    statistic = []
    py_candidate_commits = []
    py_version_bumping_commits = []
    py_update_commits = []

    for f in config_files:
        candidate_commits_path = f"{base_folder}/{f}_candidate_update_commits.csv"
        candidate_commits = pd.read_csv(
            candidate_commits_path, low_memory=False, keep_default_na=False
        )
        num_candidate_commits = candidate_commits["commit"].nunique()
        num_cfg_blobs = pd.concat(
            [candidate_commits["new blob"], candidate_commits["old blob"]]
        ).nunique()
        version_bumping_commits_path = f"{base_folder}/{f}_version_bumping_commits.csv"
        version_bumping_commits = pd.read_csv(
            version_bumping_commits_path, low_memory=False, keep_default_na=False
        )
        num_bumping_commits = version_bumping_commits["commit"].nunique()
        update_commits_path = f"{base_folder}/{f}_update_commits"
        update_commits = pd.read_csv(
            update_commits_path,
            sep=";",
            header=None,
            names=["commit", "filepath", "new_blob", "old_blob"],
            low_memory=False,
            keep_default_na=False,
        )
        num_update_commits = update_commits["commit"].nunique()
        num_code_blobs = pd.concat(
            [update_commits["new_blob"], update_commits["old_blob"]]
        ).nunique()
        if f != "pom.xml":
            py_candidate_commits.append(candidate_commits)
            py_version_bumping_commits.append(version_bumping_commits)
            py_update_commits.append(update_commits)
        statistic.append(
            [
                f,
                num_candidate_commits,
                num_cfg_blobs,
                num_bumping_commits,
                num_update_commits,
                num_code_blobs,
            ]
        )
    py_candidate_commits = pd.concat(py_candidate_commits)
    py_version_bumping_commits = pd.concat(py_version_bumping_commits)
    py_update_commits = pd.concat(py_update_commits)
    num_py_candidate_commits = py_candidate_commits["commit"].nunique()
    num_py_cfg_blobs = pd.concat(
        [py_candidate_commits["new blob"], py_candidate_commits["old blob"]]
    ).nunique()
    num_py_bumping_commits = py_version_bumping_commits["commit"].nunique()
    num_py_update_commits = py_update_commits["commit"].nunique()
    num_py_code_blobs = pd.concat(
        [py_update_commits["new_blob"], py_update_commits["old_blob"]]
    ).nunique()
    statistic.append(
        [
            "Total",
            num_py_candidate_commits,
            num_py_cfg_blobs,
            num_py_bumping_commits,
            num_py_update_commits,
            num_py_code_blobs,
        ]
    )
    return pd.DataFrame(
        statistic,
        columns=[
            "Configuration File",
            "# CU Commits",
            "# CFG Blobs",
            "# VB Commits",
            "# Update Commits",
            "# Code Blobs",
        ],
    )


count()

,Configuration File,# CU Commits,# CFG Blobs,# VB Commits,# Update Commits,# Code Blobs
0,pom.xml,31852062,63959850,6103952,1049840,27391794
1,requirements.txt,14677782,10189499,7719071,672542,6745894
2,setup.py,7622076,5525611,238418,78275,1394187
3,pyproject.toml,2725506,2045404,64965,16586,266739
4,setup.cfg,1174085,719294,29832,7886,131717
5,Total,25002699,18479562,7993598,753908,7794599


In [ ]:
from poetry.core.constraints.version.parser import parse_constraint


def has_overlap(row):
    v1 = row["version before"]
    v2 = row["version after"]
    if (v1 == "") or (v2 == ""):
        return True
    try:
        v1 = parse_constraint(v1)
        v2 = parse_constraint(v2)
        if v1.intersect(v2).is_empty():
            return False
    except:
        return True
    return True


def find_nonoverlap():
    data = []
    for f in config_files[1:]:
        version_change_commits = pd.read_csv(
            f"{base_folder}/{f}_version_changing_commits.csv",
            low_memory=False,
            keep_default_na=False,
        )
        data.append(version_change_commits)
    data = pd.concat(data, ignore_index=True)
    data = data[~data.apply(has_overlap, axis=1)]
    data.to_csv(f"{base_folder}/py_version_changing.csv", index=False)
    return data


nonfixed_changes = find_nonoverlap()

In [ ]:
from woc.local import WocMapsLocal

woc = WocMapsLocal()


def get_update_commits(sha: str):
    res = []
    try:
        fbbs = woc.get_values("c2fbb", sha)
    except:
        return []
    for f, nb, ob in fbbs:
        if not f.endswith(".py"):
            continue
        if len(nb) != 40:
            continue
        if len(ob) != 40:
            continue
        res.append((sha, f, nb, ob))
    return res

In [ ]:
from tqdm import tqdm

res = []
for sha in tqdm(nonfixed_changes["commit"].nunique()):
    res.extend(get_update_commits(sha))
res = pd.DataFrame(res, columns=["commit", "file", "new_blob", "old_blob"])
nonfixed_update_commits = res[
    ~(
        res["file"].str.contains("/setup.py")
        | (res["file"] == "setup.py")
        | res["file"].str.contains("site-packages/")
    )
].drop_duplicates()
nonfixed_update_commits.to_csv(
    "benchmark/Phase1/py_nonfixed_update_commits.csv", index=False
)

100%|██████████| 363326/363326 [03:27<00:00, 1749.41it/s]


In [64]:
print(nonfixed_update_commits["commit"].nunique(), "nonfixed update commits")

156487 nonfixed update commits


In [ ]:
nonfixed_update_packages = nonfixed_changes[
    nonfixed_changes["commit"].isin(nonfixed_update_commits["commit"])
]["package"].unique()
print(len(nonfixed_update_packages), "packages updated by nonfixed update commits")

10492 packages updated by nonfixed update commits


In [ ]:
fixed_bumping = pd.concat(
    [
        pd.read_csv(
            f"{base_folder}/{f}_version_bumping_commits.csv",
            low_memory=False,
            keep_default_na=False,
        )
        for f in config_files[1:]
    ]
)
fixed_update_commits = pd.read_csv(
    "benchmark/Phase1/py_update_commits",
    sep=";",
    header=None,
    names=["commit", "file", "new_blob", "old_blob"],
    low_memory=False,
    keep_default_na=False,
)

In [75]:
print(fixed_update_commits["commit"].nunique(), "fixed update commits")
fixed_update_packages = fixed_bumping[
    fixed_bumping["commit"].isin(fixed_update_commits["commit"])
]["package"].unique()
print(len(fixed_update_packages), "packages updated by fixed update commits")

753908 fixed update commits
25398 packages updated by fixed update commits


In [80]:
nonfixed_only_packages = list(
    set(nonfixed_update_packages) - set(fixed_update_packages)
)
print(len(nonfixed_only_packages), "packages only in nonfixed update commits")

3520 packages only in nonfixed update commits


In [90]:
nonfixed_only_commits = nonfixed_changes[
    nonfixed_changes["package"].isin(nonfixed_only_packages)
]["commit"].unique()
nonfixed_only_update_commits = nonfixed_update_commits[
    nonfixed_update_commits["commit"].isin(nonfixed_only_commits)
]